# Agent-Based Models in Finance --- Companion Notebook

Reproduces every worked numerical example in Chapter 18, `chapter18.tex`.

---

**© 2026 Wulin Suo. All rights reserved.** This notebook is a companion to the draft manuscript *AI in Finance* and is provided for personal, educational use. No part of this notebook may be reproduced, distributed, or transmitted in any form or by any means without the prior written permission of the author, except for brief quotations in a review. Contact: Wulin.Suo@Queensu.ca

## Worked Example 1: Zero-Intelligence Double Auction

In [1]:
# True valuations and costs
V_B1, V_B2 = 52.0, 48.0
C_S1, C_S2 = 45.0, 50.0

efficient_surplus = (V_B1 - C_S1)
print('Efficient allocation surplus (buyer1-seller1):', efficient_surplus)

# One simulated round of zero-intelligence random bids/asks
b1, b2 = 50.0, 30.0
a1, a2 = 47.0, 65.0

bids = sorted([b1, b2], reverse=True)
asks = sorted([a1, a2])
print('Bids (descending):', bids)
print('Asks (ascending):', asks)

trades = []
for bid, ask in zip(bids, asks):
    if bid >= ask:
        trades.append((bid, ask))
print('Trades (bid, ask):', trades)

trade_price = (b1 + a1) / 2
print('Transaction price (midpoint):', trade_price)
print('Realized surplus (true valuations):', V_B1 - C_S1)

Efficient allocation surplus (buyer1-seller1): 7.0
Bids (descending): [50.0, 30.0]
Asks (ascending): [47.0, 65.0]
Trades (bid, ask): [(50.0, 47.0)]
Transaction price (midpoint): 48.5
Realized surplus (true valuations): 7.0


### Supplementary: Is a Single Random Round Reliable?
The hand-worked example above used one particular random draw that happened to select the efficient pair. Reusable function version, plus two checks: (1) does the same seed used in the chapter text reproduce that exact outcome, and (2) how reliable is a *single* simultaneous-clearing round, averaged over many independent trials, at capturing the efficient surplus?

In [2]:
import random

def zero_intelligence_round(buyers, sellers, seed=None):
    rng = random.Random(seed)
    bids = [(rng.uniform(0, v), v) for v in buyers]
    asks = [(rng.uniform(c, 100), c) for c in sellers]
    bids.sort(key=lambda x: -x[0])
    asks.sort(key=lambda x: x[0])
    trades = []
    for (bid, val), (ask, cost) in zip(bids, asks):
        if bid >= ask:
            trades.append({'price': (bid + ask) / 2, 'surplus': val - cost})
    return trades

trades = zero_intelligence_round(buyers=[52.0, 48.0], sellers=[45.0, 50.0], seed=2)
print(trades)

[{'price': 48.912053681607716, 'surplus': 7.0}]


In [3]:
import numpy as np

np.random.seed(7)
V_B1, V_B2 = 52.0, 48.0
C_S1, C_S2 = 45.0, 50.0
efficient_surplus = V_B1 - C_S1

n_trials = 10000
captured = []
for _ in range(n_trials):
    b1 = np.random.uniform(0, V_B1)
    b2 = np.random.uniform(0, V_B2)
    a1 = np.random.uniform(C_S1, 100)
    a2 = np.random.uniform(C_S2, 100)
    buyers = sorted([(b1, V_B1), (b2, V_B2)], key=lambda x: -x[0])
    sellers = sorted([(a1, C_S1), (a2, C_S2)], key=lambda x: x[0])
    total_surplus = 0.0
    for (bid, val), (ask, cost) in zip(buyers, sellers):
        if bid >= ask:
            total_surplus += (val - cost)
    captured.append(total_surplus)

captured = np.array(captured)
print('Mean realized surplus over', n_trials, 'single-shot trials:', round(captured.mean(), 4))
print('Efficient surplus:', efficient_surplus)
print('Average efficiency captured (%):', round(100*captured.mean()/efficient_surplus, 2))
print('Fraction of trials reaching full efficient surplus:', round(np.mean(captured==efficient_surplus), 4))

Mean realized surplus over 10000 single-shot trials: 0.0615
Efficient surplus: 7.0
Average efficiency captured (%): 0.88
Fraction of trials reaching full efficient surplus: 0.0076


## Worked Example 2: Fundamentalist-Chartist Price Dynamics

In [4]:
p_f = 100.0
alpha = 0.3
beta = 1.0
lam = 1.5

def step(p_prev, p_prev2, nf, nc):
    d_f = alpha * (p_f - p_prev)
    d_c = beta * (p_prev - p_prev2)
    agg = nf * d_f + nc * d_c
    p_new = p_prev + lam * agg
    return p_new, d_f, d_c, agg

print('--- Scenario A: fundamentalist-dominated (n_f=0.8, n_c=0.2) ---')
p_prev2, p_prev = 93.0, 95.0
nf, nc = 0.8, 0.2
for t in range(3):
    p_new, d_f, d_c, agg = step(p_prev, p_prev2, nf, nc)
    print(f't={t}: p_t={p_prev:.3f}  d_f={d_f:.3f}  d_c={d_c:.3f}  agg={agg:.3f}  p_t+1={p_new:.3f}')
    p_prev2, p_prev = p_prev, p_new

--- Scenario A: fundamentalist-dominated (n_f=0.8, n_c=0.2) ---
t=0: p_t=95.000  d_f=1.500  d_c=2.000  agg=1.600  p_t+1=97.400
t=1: p_t=97.400  d_f=0.780  d_c=2.400  agg=1.104  p_t+1=99.056
t=2: p_t=99.056  d_f=0.283  d_c=1.656  agg=0.558  p_t+1=99.893


In [5]:
print('--- Scenario B: pure chartist, no fundamentalists (n_f=0.0, n_c=1.0) ---')
p_prev2, p_prev = 93.0, 95.0
nf, nc = 0.0, 1.0
for t in range(3):
    p_new, d_f, d_c, agg = step(p_prev, p_prev2, nf, nc)
    print(f't={t}: p_t={p_prev:.3f}  p_t+1={p_new:.3f}  (period change = {p_new - p_prev:.3f})')
    p_prev2, p_prev = p_prev, p_new

--- Scenario B: pure chartist, no fundamentalists (n_f=0.0, n_c=1.0) ---
t=0: p_t=95.000  p_t+1=98.000  (period change = 3.000)
t=1: p_t=98.000  p_t+1=102.500  (period change = 4.500)
t=2: p_t=102.500  p_t+1=109.250  (period change = 6.750)


## Emergent Stylized Facts: A Regime-Switching Simulation

Extends the fundamentalist-chartist model to 500 periods with two regimes: a calm, fundamentalist-heavy regime and a volatile, chartist-heavy regime. As a simple proxy for the performance-based switching Brock and Hommes formalize, the market switches into the volatile regime whenever the most recent price move exceeds a threshold (a large recent move is exactly the situation in which a chartist strategy would just have outperformed a fundamentalist one), and reverts to the calm regime otherwise.

In [6]:
import numpy as np
from scipy import stats

np.random.seed(42)

p_f = 100.0
alpha = 0.3
beta_switch = 1.1
lam_switch = 1.0

nf_calm, nc_calm = 0.85, 0.15
nf_vol, nc_vol = 0.25, 0.75
sigma_calm = 0.15
sigma_vol = 0.50
switch_threshold = 0.25

T = 500
prices = [95.0, 95.0]
regimes = []

for t in range(2, T + 2):
    p_prev, p_prev2 = prices[-1], prices[-2]
    recent_move = p_prev - p_prev2
    if abs(recent_move) > switch_threshold:
        nf, nc, sigma_e = nf_vol, nc_vol, sigma_vol
        regime = 'volatile'
    else:
        nf, nc, sigma_e = nf_calm, nc_calm, sigma_calm
        regime = 'calm'
    d_f = alpha * (p_f - p_prev)
    d_c = beta_switch * (p_prev - p_prev2)
    shock = np.random.normal(0, sigma_e)
    p_new = p_prev + lam_switch * (nf * d_f + nc * d_c) + shock
    prices.append(p_new)
    regimes.append(regime)

prices = np.array(prices)
regimes = np.array(regimes)
returns = np.diff(prices)
returns_aligned = returns[1:]  # align with regimes (first return has no regime label)

n_calm = np.sum(regimes == 'calm')
n_vol = np.sum(regimes == 'volatile')
calm_ret = returns_aligned[regimes == 'calm']
vol_ret = returns_aligned[regimes == 'volatile']

print('Price range over 500 periods:', prices.min().round(3), 'to', prices.max().round(3))
print('Periods in calm regime:', n_calm, '  Periods in volatile regime:', n_vol)
print('Variance of returns, calm regime:    ', round(calm_ret.var(), 4))
print('Variance of returns, volatile regime:', round(vol_ret.var(), 4))
print('Variance ratio (volatile / calm):    ', round(vol_ret.var() / calm_ret.var(), 3))
print()
print('Full-series excess kurtosis (Fisher, Gaussian=0):', round(stats.kurtosis(returns), 3))
print('Full-series kurtosis (Pearson, Gaussian=3):      ', round(stats.kurtosis(returns, fisher=False), 3))
print()
sq = returns ** 2
ac1 = np.corrcoef(sq[:-1], sq[1:])[0, 1]
ac5 = np.corrcoef(sq[:-5], sq[5:])[0, 1]
print('Autocorrelation of squared returns, lag 1:', round(ac1, 3))
print('Autocorrelation of squared returns, lag 5:', round(ac5, 3))

Price range over 500 periods: 90.608 to 108.546
Periods in calm regime: 216   Periods in volatile regime: 284
Variance of returns, calm regime:     0.1569
Variance of returns, volatile regime: 1.121
Variance ratio (volatile / calm):     7.143

Full-series excess kurtosis (Fisher, Gaussian=0): 2.234
Full-series kurtosis (Pearson, Gaussian=3):       5.234

Autocorrelation of squared returns, lag 1: 0.761
Autocorrelation of squared returns, lag 5: 0.297


## Worked Example 3: Eisenberg-Noe Contagion Cascade

In [7]:
# Baseline (no shock): A -> B $50, B -> C $40, C has no obligations
external_baseline = {'A': 60.0, 'B': 10.0, 'C': 5.0}
owed = {'A': 50.0, 'B': 40.0, 'C': 0.0}  # what each bank owes

def clear_chain(external):
    # Simple chain A->B->C: resolve in payment order
    resources_A = external['A']
    paid_A = min(resources_A, owed['A'])
    resources_B = external['B'] + paid_A
    paid_B = min(resources_B, owed['B'])
    resources_C = external['C'] + paid_B
    paid_C = min(resources_C, owed['C'])
    net_worth = {
        'A': resources_A - paid_A,
        'B': resources_B - paid_B,
        'C': resources_C - paid_C,
    }
    paid = {'A': paid_A, 'B': paid_B, 'C': paid_C}
    received = {'A': 0.0, 'B': paid_A, 'C': paid_B}
    return paid, received, net_worth

print('--- Baseline (no shock) ---')
paid, received, net_worth = clear_chain(external_baseline)
for bank in ['A', 'B', 'C']:
    print(f'{bank}: received={received[bank]:.2f}  paid={paid[bank]:.2f}  net worth={net_worth[bank]:.2f}')
print('Total system net worth:', sum(net_worth.values()))

--- Baseline (no shock) ---
A: received=0.00  paid=50.00  net worth=10.00
B: received=50.00  paid=40.00  net worth=20.00
C: received=40.00  paid=0.00  net worth=45.00
Total system net worth: 75.0


In [8]:
# Shock: A's external assets fall from $60 to $15 (a $45 loss)
external_shocked = {'A': 15.0, 'B': 10.0, 'C': 5.0}

print('--- After a $45 shock to Bank A\'s external assets ---')
paid, received, net_worth = clear_chain(external_shocked)
for bank in ['A', 'B', 'C']:
    default_flag = ' (DEFAULT)' if paid[bank] < owed[bank] else ''
    print(f'{bank}: received={received[bank]:.2f}  paid={paid[bank]:.2f} of {owed[bank]:.2f} owed{default_flag}  net worth={net_worth[bank]:.2f}')
print('Total system net worth:', sum(net_worth.values()))
print('Total system-wide loss:', 75.0 - sum(net_worth.values()))

--- After a $45 shock to Bank A's external assets ---
A: received=0.00  paid=15.00 of 50.00 owed (DEFAULT)  net worth=0.00
B: received=15.00  paid=25.00 of 40.00 owed (DEFAULT)  net worth=0.00
C: received=25.00  paid=0.00 of 0.00 owed  net worth=30.00
Total system net worth: 30.0
Total system-wide loss: 45.0


## Advanced Topics: Chain versus Hub-and-Spoke Contagion

In [9]:
# Hub-and-spoke network: H owes $30 each to P1, P2, P3; same $45 shock as the chain example
owed_each = 30.0
n_peripherals = 3
total_owed = owed_each * n_peripherals
external_H_baseline = 100.0
external_Pi = 5.0

def clear_star(external_H):
    resources_H = external_H
    paid_H = min(resources_H, total_owed)
    net_H = resources_H - paid_H
    per_Pi_received = paid_H * (owed_each / total_owed)
    net_Pi = external_Pi + per_Pi_received
    total = net_H + n_peripherals * net_Pi
    return net_H, net_Pi, per_Pi_received, paid_H, total

print('--- Baseline ---')
net_H, net_Pi, per_Pi_received, paid_H, total = clear_star(external_H_baseline)
print(f'H: net worth={net_H:.2f}   Each P_i: net worth={net_Pi:.2f}   Total={total:.2f}')

print('--- After a $45 shock to H (external assets 100 -> 55) ---')
net_H_s, net_Pi_s, per_Pi_received_s, paid_H_s, total_s = clear_star(external_H_baseline - 45.0)
recovery = paid_H_s/total_owed*100
pct_loss_Pi = (net_Pi-net_Pi_s)/net_Pi*100
print(f'H: recovery={recovery:.1f}%  net worth={net_H_s:.2f} (DEFAULT)')
print(f'Each P_i: received={per_Pi_received_s:.2f}  net worth={net_Pi_s:.2f}  loss={pct_loss_Pi:.1f}%')
print(f'Total system net worth: {total_s:.2f}   Total loss: {total-total_s:.2f}')

--- Baseline ---
H: net worth=10.00   Each P_i: net worth=35.00   Total=115.00
--- After a $45 shock to H (external assets 100 -> 55) ---
H: recovery=61.1%  net worth=0.00 (DEFAULT)
Each P_i: received=18.33  net worth=23.33  loss=33.3%
Total system net worth: 70.00   Total loss: 45.00


### Stability Threshold Verification: n_c* = 1/(lambda*beta)

In [10]:
p_f = 100.0
alpha, beta, lam = 0.3, 1.0, 1.5
nc_star = 1 / (lam * beta)
print('Theoretical critical threshold n_c* =', round(nc_star, 4))

def simulate_path(nc, steps):
    nf = 1 - nc
    p_prev2, p_prev = 93.0, 95.0
    path = [p_prev2, p_prev]
    for t in range(steps):
        d_f = alpha * (p_f - p_prev)
        d_c = beta * (p_prev - p_prev2)
        p_new = p_prev + lam * (nf * d_f + nc * d_c)
        p_prev2, p_prev = p_prev, p_new
        path.append(p_new)
    return path

print()
print('n_c = 0.60 (below threshold, should converge):')
path_below = simulate_path(0.60, 60)
print([round(path_below[i], 2) for i in [0, 10, 20, 30, 40, 50, 60]])

print()
print('n_c = 0.70 (above threshold, should diverge):')
path_above = simulate_path(0.70, 15)
print([round(path_above[i], 2) for i in range(0, 16, 3)])

Theoretical critical threshold n_c* = 0.6667

n_c = 0.60 (below threshold, should converge):
[93.0, 100.02, 102.43, 99.0, 99.56, 100.53, 99.94]

n_c = 0.70 (above threshold, should diverge):
[93.0, 100.99, 109.08, 107.85, 97.26, 88.2]


## ABM versus MARL: A Learning Seller in the Double Auction

In [11]:
import random
random.seed(11)

cost = 40.0
ask_high = 58.0
ask_low = 47.0
valuations = [45, 50, 55, 60, 65]

def true_ev(ask):
    trades = [v for v in valuations if v >= ask]
    p_trade = len(trades) / len(valuations)
    return p_trade * (ask - cost)

print('True E[profit] Arm H:', true_ev(ask_high))
print('True E[profit] Arm Lo:', true_ev(ask_low))

seq = [random.choice(valuations) for _ in range(12)]
print('Buyer valuation sequence:', seq)

history = {'H': [], 'Lo': []}
arm_schedule_forced = ['H', 'Lo', 'H', 'Lo']

for i, v in enumerate(seq):
    if i < 4:
        arm = arm_schedule_forced[i]
    else:
        avgH = sum(history['H']) / len(history['H']) if history['H'] else float('-inf')
        avgLo = sum(history['Lo']) / len(history['Lo']) if history['Lo'] else float('-inf')
        arm = 'H' if avgH >= avgLo else 'Lo'
    ask = ask_high if arm == 'H' else ask_low
    profit = (ask - cost) if v >= ask else 0.0
    history[arm].append(profit)
    avgH_after = sum(history['H'])/len(history['H']) if history['H'] else float('nan')
    avgLo_after = sum(history['Lo'])/len(history['Lo']) if history['Lo'] else float('nan')
    print(f'Round {i+1:2d}: value={v}  arm={arm:2s}  profit={profit:5.1f}  avgH={avgH_after:5.2f}  avgLo={avgLo_after:5.2f}')

True E[profit] Arm H: 7.2
True E[profit] Arm Lo: 5.6000000000000005
Buyer valuation sequence: [60, 65, 60, 60, 65, 65, 50, 50, 65, 60, 65, 50]
Round  1: value=60  arm=H   profit= 18.0  avgH=18.00  avgLo=  nan
Round  2: value=65  arm=Lo  profit=  7.0  avgH=18.00  avgLo= 7.00
Round  3: value=60  arm=H   profit= 18.0  avgH=18.00  avgLo= 7.00
Round  4: value=60  arm=Lo  profit=  7.0  avgH=18.00  avgLo= 7.00
Round  5: value=65  arm=H   profit= 18.0  avgH=18.00  avgLo= 7.00
Round  6: value=65  arm=H   profit= 18.0  avgH=18.00  avgLo= 7.00
Round  7: value=50  arm=H   profit=  0.0  avgH=14.40  avgLo= 7.00
Round  8: value=50  arm=H   profit=  0.0  avgH=12.00  avgLo= 7.00
Round  9: value=65  arm=H   profit= 18.0  avgH=12.86  avgLo= 7.00
Round 10: value=60  arm=H   profit= 18.0  avgH=13.50  avgLo= 7.00
Round 11: value=65  arm=H   profit= 18.0  avgH=14.00  avgLo= 7.00
Round 12: value=50  arm=H   profit=  0.0  avgH=12.60  avgLo= 7.00


## Method of Simulated Moments: Sweeping the Switching Threshold

In [12]:
import numpy as np
from scipy import stats

def run_sim(threshold, seed=42):
    np.random.seed(seed)
    p_f = 100.0
    alpha, beta_switch, lam_switch = 0.3, 1.1, 1.0
    nf_calm, nc_calm = 0.85, 0.15
    nf_vol, nc_vol = 0.25, 0.75
    sigma_calm, sigma_vol = 0.15, 0.50
    T = 500
    prices = [95.0, 95.0]
    for t in range(2, T + 2):
        p_prev, p_prev2 = prices[-1], prices[-2]
        recent = p_prev - p_prev2
        if abs(recent) > threshold:
            nf, nc, sigma_e = nf_vol, nc_vol, sigma_vol
        else:
            nf, nc, sigma_e = nf_calm, nc_calm, sigma_calm
        d_f = alpha * (p_f - p_prev)
        d_c = beta_switch * (p_prev - p_prev2)
        shock = np.random.normal(0, sigma_e)
        p_new = p_prev + lam_switch * (nf * d_f + nc * d_c) + shock
        prices.append(p_new)
    prices = np.array(prices)
    returns = np.diff(prices)
    return stats.kurtosis(returns, fisher=False)

for th in [0.15, 0.20, 0.25, 0.30, 0.35]:
    k = run_sim(th)
    print(f'threshold={th:.2f}  Pearson kurtosis={k:.3f}')

threshold=0.15  Pearson kurtosis=3.820
threshold=0.20  Pearson kurtosis=3.914
threshold=0.25  Pearson kurtosis=5.234
threshold=0.30  Pearson kurtosis=6.622
threshold=0.35  Pearson kurtosis=11.640
